In [ ]:
# Libraries
from pypdf import PdfReader
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown


In [ ]:
# Load variables
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# Intance reader
url = r"c:\Users\PatricioGarcia\Desktop\Pato\Programación\Proyectos\abap_requirement_agent\data\EF_GB-005 Formato de Validación de carga_v1.pdf"
reader = PdfReader()

# Get requirement pdf txt
requirement_txt = ""
for page in reader.pages:
    requirement_txt += page.extract_text() or ""

print(requirement_txt)

 
 
 
 
 
 
 
 
EF_EWM-005  
Imprimible: Formato de validación de carga  
IBP BAFAR  
IBP BAFAR EWM 
 
V 1.0 
 
 
 
 
 
 
 
 
Especificación Funcional 
 
   
Especificación Funcional (Módulo SAP)  Página 1 
 
 
Índice 
1. Control de Documento .......................................................................................................................... 2 
1.1. Detalle de la Especificación ........................................................................................................... 2 
1.2. Objetivo ......................................................................................................................................... 2 
1.3. Control de Versiones ..................................................................................................................... 2 
2. Definición General ................................................................................................................................. 2 
2.1. Razón para desarrollar .

In [ ]:
# Create model instance
openai = OpenAI()

In [ ]:
# Create prompts
system_prompt = """ 

                    Sos un analista funcional y técnico senior en SAP ABAP con experiencia en proyectos reales de implementación y mantenimiento.

                    Tu rol es analizar requerimientos funcionales y técnicos y evaluar si son suficientemente completos, claros y desarrollables.

                    Siempre trabajás con criterio profesional, siendo crítico pero constructivo.

                    Tenés en cuenta buenas prácticas de SAP, calidad de desarrollo y riesgos de implementación.

                    ---

                    Tu tarea cuando recibís un requerimiento:

                    1. Generar un resumen ejecutivo claro (máximo 10 líneas).
                    2. Identificar qué partes del requerimiento están bien definidas.
                    3. Detectar información faltante o incompleta.
                    4. Detectar ambigüedades o posibles interpretaciones múltiples.
                    5. Identificar riesgos técnicos o funcionales.
                    6. Proponer preguntas concretas para el funcional o usuario.
                    7. Evaluar el nivel de completitud del requerimiento:
                    - Completo
                    - Parcialmente completo
                    - Incompleto

                    ---

                    Checklist mínima para evaluar:

                    - Objetivo del requerimiento
                    - Alcance (qué incluye y qué no)
                    - Proceso de negocio afectado
                    - Transacción, programa o desarrollo involucrado
                    - Tablas y estructuras de datos
                    - Campos específicos afectados
                    - Reglas de negocio
                    - Validaciones
                    - Mensajes de error / warning
                    - Datos de entrada y salida
                    - Interfaces o integraciones impactadas
                    - Jobs o procesos batch
                    - Autorizaciones
                    - Casos de prueba / criterios de aceptación
                    - Casos borde o excepciones

                    ---

                    Formato de respuesta obligatorio:

                    ## Resumen
                    ...

                    ## Puntos claros
                    ...

                    ## Faltantes
                    ...

                    ## Ambigüedades
                    ...

                    ## Riesgos
                    ...

                    ## Preguntas clave
                    ...

                    ## Evaluación final
                    (Completo / Parcialmente completo / Incompleto)

                    ---

                    Reglas:

                    - No inventar información que no esté en el documento.
                    - Si algo no está, marcarlo explícitamente.
                    - Ser específico y concreto (no genérico).
                    - Escribir en español claro y profesional. """

user_promt = f"""
                Analizá el siguiente requerimiento ABAP.

                Quiero que hagas un análisis profundo, no superficial.

                En particular:
                - No asumas nada que no esté explícito
                - Si falta información, marcala claramente
                - Si algo es ambiguo, explicá por qué
                - Pensá como desarrollador ABAP que tiene que implementarlo

                Además:
                - Sé específico, evitá generalidades
                - Usá el checklist definido en el sistema
                - Priorizá problemas reales de implementación

                Requerimiento:
                
                {requirement_txt}
                """



In [ ]:
# Call API
response = openai.chat.completions.create(
    model='gpt-5-mini',
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content" : user_promt}
    ]
)



In [29]:
# Output
Markdown(response.choices[0].message.content)

## Resumen
Revisión funcional-técnica del requerimiento "Imprimible: Formato de validación de carga" para SAP EWM. El documento define el objetivo, alcance, mapeo de campos y la detonación vía PPF al crear WTs en /SCWM/TU. Contiene instrucciones de consulta a tablas EWM y llamadas a ECC (FM en grupo ZGFIBP63_TM). Sin embargo faltan detalles técnicos clave para implementar: especificación del formulario (tecnología), detalles de PPF (objeto/evento/condición), nombres/atributos exactos de campos y tablas (algunos parecen custom/incorrectos), tratamiento de errores y rendimiento, y criterios de pruebas y reimpresión. Evaluación: Parcialmente completo.

## Puntos claros
- Objetivo: generar automáticamente un imprimible (hoja carta) "ENTREGA A CLIENTES" por cada entrega asociada a una TU cuando se crea la WT de carga en /SCWM/TU.
- Alcance funcional: documento informativo, no modifica datos, generado por PPF al crear tareas de carga; impresión física en impresora asignada.
- Se indican las 3 secciones del formulario (cabecera, cuerpo, acuerdos/firmas) y el formato general (hoja carta, logo).
- Listado de tablas/consultas propuestas para la mayoría de los campos (por ejemplo /SCWM/TUNIT, /SCWM/TU_DLV, /SCDL/DB_REFDOC, /SCDL/DB_DATE, /SCDL/DB_PROCI_O, /SCDL/DB_BPLOC, /SCWM/HUHDR, /SAPAPO/MATKEY).
- Existe la instrucción explícita de crear un Function Module en ECC dentro del function group ZGFIBP63_TM para consultar datos en ECC (VBAK, VEDA, KNVA).
- Se requiere conversión de fecha a formato DD. MM.AAAA y timezone = UTC-6.
- Requerimiento para poder reimprimir el documento.

## Faltantes (información que no está y es necesaria)
- Tecnología del formulario: no se especifica si será Smart Forms, SAPscript, Adobe Forms (PDF), u otra. Esto define cómo se diseñan y transportan los formularios, la impresión y la paginación.
- Nombre técnico del PPF object / evento / action / condition record a parametrizar; qué objeto de negocio (p.ej. /SCWM/TU o tipo de objeto) y cuál evento será usado exactamente.
- Usuario/sistema bajo el que se ejecuta la PPF (implicación de autorizaciones y spool owner).
- RFC destination / nombre del sistema ECC y contrato RFC; especificación de si el FM será remoto (RFC) y cómo se llamará desde EWM (sin detalle).
- Especificación exacta de los campos en las tablas (longitud, tipo de dato, presencia segura): muchas columnas referidas (ej. TOP_HUIDENT, GUID_HU, NWEIGHT, VSOLM, WEIGHT, TU_NUM_EXT, TU_NUM, DOCID, ITEMID) deben confirmarse en el sistema EWM actual.
- Confirmación de la existencia y firma de FMs estándar usados: BUPA_ADDRESS_GET_DETAIL (¿existe o es custom?), y del FM a crear (firma, parámetros de entrada/salida).
- Especificación de comportamiento cuando ECC no responde: actualmente se indica que quedarán vacíos, pero no se define logging, reintentos, mensajes a usuario ni si la impresión debe abortar o continuar.
- Reglas de negocio para casos de: entregas parcialmente pickeadas, HUs sin peso, múltiples lotes por posición, posiciones con múltiples HUs (cómo agrupar/sumar), unidades de medida distintas (conversión).
- Especificación precisa de la lógica "TOTAL + MATERIAL": ¿qué se suma? ¿es un subtotal por línea o un total al final? Cómo formatear.
- Paginación y numeración: la regla de 1 imprimible por entrega contradice la nota de paginación 1/3 por TU; no se define si la numeración es páginas por entrega o páginas por TU.
- Parámetros de impresión: nombre de impresora(s) por almacén, mapping EWM->device, tray, duplex, orientación, margins; spool parameters y si se usará driver SAP spool o impresión directa.
- Reimpresión: procedimiento (transacción/selección), restricciones (quién puede reimprimir), número máximo de reimpresiones, registro de reimpresiones.
- Pruebas: casos de prueba concretos con datos de ejemplo (TU con 1 entrega; TU con N entregas; posiciones multiple lotes; ECC indisponible; pick parcial; gran volumen).
- Mensajes de error / warning: no se listan textos, códigos ni comportamiento ante fallos.
- Autorizaciones necesarias: roles/objetos de autorización requeridos para disparar PPF, ejecutar RFC, imprimir.
- Localización/idioma y variantes: el documento muestra textos en español; no se indica si habrá otros idiomas.
- Formato exacto de fecha/hora (ejemplo inconsistencias: DD. MM.AAAA con punto y espacio; en otro punto DD/MM/AAAA) y uso de zona horaria en conversión (UTC-6 DST?).
- Regla de eliminación de duplicados: no especifica criterio (por ejemplo, eliminar por DOCID únicamente o DOCID+ITEMID).
- Performance/volumen estimado (número típico de TUs / entregas / líneas / HUs por ejecución) para dimensionar el diseño y evitar timeouts.

## Ambigüedades (posibles interpretaciones múltiples)
- "1 imprimible por cada entrega que esté asignada a la TU" vs "Página enumerada como 1/3": ¿se imprime un documento por entrega (cada uno puede tener múltiples páginas) pero la paginación 1/3 se refiere al total de páginas de la TU o al total de entregas? Ambigüedad en el alcance de la numeración.
- Campo "Total: TOTAL + el valor extraído en el punto 2 MATERIAL": no queda claro si "TOTAL" es texto fijo seguido del código de material, o si debe mostrar suma (p. ej. total de piezas o kilos) del material. Texto es contradictorio con campo "Total".
- Uso de tablas SCDL/... y nombres de campos: prefijo /SCDL/ y campos como DB_REFDOC, DB_PROCI_O, DB_DATE parecen Z- o Y- objetos (custom). No está claro si las columnas mencionadas existen en el sistema actual con esos nombres. Puede haber confusión en el mapping técnico.
- Llamadas a ECC:
  - Tablas VBAK ZZ_ROUTE y VEDA (VBEGDAT & VENDDAT): ZZ_ROUTE es campo Z (custom) — confirmado como custom; VEDA no es tabla estándar en la nomenclatura común (más habitual VEDA no existe en ERP estándar), por lo que puede tratarse de un objeto custom o mal escrito (p. ej. VEDA vs VEDA?).
  - Uso del FM BUPA_ADDRESS_GET_DETAIL: no es un FM estándar conocido con ese exacto nombre (existe BUPA_ADDRESS_GET/GET_DETAIL?), por lo que puede ser custom.
- Conversión de fecha/hora:
  - Indican timezone = UTC-6 sin especificar DST ni cómo mapear campos TSTFR (tipo timestamp) a la zona. EWM guarda fechas/horas en UTC o en formato de usuario; es ambigua la manipulación.
- "El formulario deberá ser impreso, una vez que todas las entregas estén pickeadas. Esta validación no será realizada por el sistema." Contradicción: el disparador es la creación de WTs en /SCWM/TU (cuando generalmente las WTs se crean después del pick); pero si la validación no será automática, existe riesgo de imprimir cuando no todo esté pickeado — quién controla esto?
- Campo "Solicitante" y "Destinatario": en punto 6 piden PARTY_ROLE = SOTPRT para obtener Solicitante y en 11 PARTY_ROLE = STPRT para Destinatario; después dicen "Solicitante + su descripción" y para descripción piden BUT000 lookup con LGNUM=PARTNER lo cual no tiene sentido técnico (BUT000 no tiene LGNUM). Ambigüedad sobre tablas/columnas a usar para descripción.

## Riesgos técnicos y funcionales
- Integridad de datos: muchas tablas referidas parecen custom; si sus estructuras cambian o no existen, el desarrollo fallará.
- Disponibilidad ECC: llamar FMs a ECC por cada impresión puede generar latencia o fallos; requisito dice que si ECC cae los campos se quedarán vacíos — esto puede degradar la utilidad del documento. Además no hay plan de retry o cache, ni degradación controlada.
- Performance: si se imprimen N documentos al crear WTs (p. ej. TU con 50 entregas, cada una con 100 posiciones y líneas HU), las llamadas a tablas y a ECC pueden causar timeouts o impactar el proceso de creación de WT. Ejecutar PPF en modo síncrono podría bloquear la transacción /SCWM/TU.
- PPF design incorrecto: si la acción PPF dispara impresión bajo el mismo proceso de creación de WT sin buffering o asincronía, habrá riesgo de bloqueo y de errores de spool. No se especifica si la impresión se debe lanzar en background o asíncrona.
- Dependencia de impresoras: asume impresoras correctamente configuradas; si no lo están, el documento fallará. No hay manejo de fallback (email, spool save).
- Autorizaciones: falta detalle de quién podrá reimprimir o lanzar PPF; riesgo de impresión indiscriminada por usuarios no autorizados.
- Formato y consistencia: sin especificación de tecnología del formulario, reproducción exacta del formato actual del cliente puede fallar (fuentes, logotipo, márgenes, códigos de barras si existen).
- Errores de cálculo: campos de kilos/piezas y suma (NWEIGHT, WEIGHT, VSOLM) pueden duplicarse o mezclarse si la relación entre TU_DLV → TOP_HUIDENT → HUHDR no es correcta en la interpretación.
- Manejo de lotes múltiples por pos.: el formato visual no está definido; riesgo de overflow de campo o de perder trazabilidad.
- Paginación y conteo: sin regla clara podría mostrarse numeración errónea (p.ej. 1/1 por entrega cuando se espera 1/3 por TU).

## Preguntas clave (para el funcional / usuario)
1. Tecnología del formulario: ¿SmartForms, SAPscript, Adobe Forms, o PDF externo? ¿Existe un ejemplo del formato actual (archivo .pdf/.doc) con márgenes y tipografías para replicar?
2. Confirmar comportamiento de paginación: ¿la numeración "Página 1/3" debe referirse a las páginas totales generadas por la TU o por la entrega individual? ¿Desean numeración por entrega (p.ej. 1/1, 1/2) o consolidada por TU?
3. ¿Desean que la impresión sea síncrona (mientras se crea WT) o asíncrona (job en background)? ¿Es aceptable un proceso asíncrono para evitar bloqueo de /SCWM/TU?
4. Nombre técnico del objeto PPF, evento y condición que se debe parametrizar (o solicitan que definamos la configuración PPF recomendada)? ¿Existe ya un status que se active al "Crear WTs" donde acoplar la acción?
5. ¿Cuál será la impresora por defecto o la lógica de determinación (por almacén/usuario/handler)? ¿Proporcionarán los nombres de device/printer para QA/PROD?
6. Reimpresión: ¿qué transacción/proceso debe permitirl/la reimpresión y quién tiene autorización? ¿Registrar la reimpresión en log?
7. ECC: ¿confirmáis que el FM a crear en ECC será expuesto como RFC remoto y que existirá una RFC destination configurada en EWM? ¿Cuál será el nombre del FM y su firma esperada (input: lista REFDOCNO; output: ZZ_ROUTE, VEDA dates, KNVA info)?
8. Validación de campos y tablas EWM: ¿podéis confirmar en el ambiente actual los nombres exactos y existencia de columnas referidas: TU_NUM_EXT, TU_NUM, DOCID, ITEMID, TOP_HUIDENT, GUID_HU, NWEIGHT, VSOLM, WEIGHT, BATCHNO, UOM, QTY?
9. BUPA_ADDRESS_GET_DETAIL: ¿este FM existe en vuestro sistema estándar o es custom? En caso de no existir, ¿permitís usar otra FM estándar o realizar read de tablas BUT000/BUT020/BUT1_* para dirección?
10. Tabla VEDA y VBAK.ZZ_ROUTE y campos en KNVA: confirmar que las tablas y campos en ECC existen y son accesibles por el FM; indicar ejemplo de datos.
11. Total: explicar exactamente qué debe contener el campo "Total": ¿texto fijo + material? ¿suma de piezas/kgs por entrega? Dar ejemplo de salida esperada.
12. Formato de fecha/hora: confirmar formato deseado (ejemplo: DD.MM.AAAA ó DD/MM/AAAA) y confirmación de timezone UTC-6 incluyendo manejo de horario de verano.
13. Comportamiento si ECC no responde: ¿debe fallar la impresión, debe imprimirse con campos en blanco, o reintentar y registrar incidente?
14. Casos de prueba: proporcionar al menos 3 escenarios con datos (TU con 1 entrega; TU con 3 entregas; TU con entrega con múltiples lotes y HUs) para validar la impresión.
15. ¿Desean incluir número de material + descripción en "Total" o "Total" se refiere a sumario final? (repetición, se requiere clarificar)
16. Logos: ¿entregarán el .jpg final y versión para producción (resolución) y permisos de uso?
17. ¿Existe policy para privacidad/datos en impresión (p.ej. si documentos impresos contienen datos confidenciales)?

## Evaluación final
Parcialmente completo

Motivo: el documento contiene el objetivo, alcance general y mapeo de la mayoría de campos, además la indicación de llamar a ECC vía FM. Sin embargo faltan detalles técnicos críticos (tecnología de formulario, PPF object/event preciso, parámetros de impresión, confirmación de tablas/campos y FMs, manejo de errores y rendimiento, reglas de paginación y reimpresión, pruebas con datos). Estos elementos son necesarios para un desarrollo ABAP/EWM seguro y reproducible.

---

Recomendación inmediata para avanzar:
- Solicitar respuestas a las preguntas listadas y proveer ejemplos de salida del formato actual (archivo de muestra).
- Validar en el sistema EWM los nombres exactos de tablas y campos mencionados.
- Definir la tecnología del formulario y la estrategia de impresión (síncrona/asíncrona).
- Definir la firma del FM ECC y configurar RFC destination de prueba.
- Preparar casos de prueba detallados y volúmenes estimados para dimensionar performance.